# YOLO25n on VOC — V1: Setup, Dataset, Class Alignment, Baseline

**Pipeline (this notebook):**
1. Environment setup (Drive mount, install, GPU check)
2. Download VOC (Ultralytics built-in, auto-converted to YOLO format)
3. Inspect dataset vs. what the model expects
4. Align VOC's 20 classes to the model's native COCO 80-class space
5. Run a **zero-shot baseline** evaluation (no training yet)

**Deferred to V2** (not in this notebook): fine-tuning on VOC, native 20-class
validation, and a baseline-vs-fine-tuned comparison report.

**Design notes (agreed before implementation):**
- Baseline uses the model's **native 80-class head** — VOC ground-truth labels
  are remapped to COCO indices, not the other way around. No model surgery.
- The raw VOC dataset itself stays **local to the Colab session** (ephemeral,
  cheap to re-download via the same cell). Only **outputs** (run logs, weights,
  reports) are persisted to Google Drive, to avoid syncing a multi-GB dataset
  to Drive on every run.


## 1. Environment Setup

In [ ]:
# ── Mount Drive & create project structure (this is where OUTPUTS persist) ──
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/yolo25n_voc'
for sub in ['runs', 'configs', 'reports']:
    os.makedirs(f'{PROJECT_ROOT}/{sub}', exist_ok=True)

print(f'Project root (persisted to Drive): {PROJECT_ROOT}')


In [ ]:
# ── Install Ultralytics ──
!pip install -q ultralytics

import ultralytics
ultralytics.checks()


In [ ]:
# ── Verify GPU runtime ──
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > select a GPU, then re-run."
)
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Dataset — Download VOC

Using Ultralytics' built-in `VOC.yaml`, which auto-downloads Pascal VOC
2007+2012 and converts annotations to YOLO `.txt` format on first use.
`check_det_dataset` triggers this without needing a full `train()`/`val()` call.


In [ ]:
# ── Download / locate VOC (auto-converts to YOLO format, cached locally) ──
from ultralytics.data.utils import check_det_dataset

voc_data = check_det_dataset('VOC.yaml')

print(f"Classes (nc={voc_data['nc']}):")
print(voc_data['names'])
print(f"\nTrain path(s): {voc_data['train']}")
print(f"Val path(s):   {voc_data['val']}")


### 2a. Inspect dataset — counts

In [ ]:
from pathlib import Path

def count_images_and_labels(img_dir):
    img_dir = Path(img_dir)
    label_dir = Path(str(img_dir).replace('images', 'labels'))
    images = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    labels = list(label_dir.glob('*.txt'))
    return len(images), len(labels)

for split in ['train', 'val']:
    paths = voc_data[split]
    paths = paths if isinstance(paths, list) else [paths]
    for p in paths:
        n_img, n_lbl = count_images_and_labels(p)
        print(f"{split:5s}: {p}  ->  {n_img} images, {n_lbl} label files")


### 2b. Inspect dataset — visualize a few samples with decoded boxes

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

def show_samples(img_dir, names, n=4, seed=0):
    img_dir = Path(img_dir if not isinstance(img_dir, list) else img_dir[0])
    label_dir = Path(str(img_dir).replace('images', 'labels'))
    random.seed(seed)
    img_paths = random.sample(list(img_dir.glob('*.jpg')), n)

    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    for ax, img_path in zip(axes, img_paths):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        label_path = label_dir / f'{img_path.stem}.txt'
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                cls, xc, yc, bw, bh = map(float, line.split())
                x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
                x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(img, names[int(cls)], (x1, max(y1 - 5, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(img_path.name, fontsize=8)
    plt.tight_layout()
    plt.show()

show_samples(voc_data['train'], voc_data['names'])


## 3. Model — Inspect what `yolo25n` expects

In [ ]:
# NOTE: adjust the weights filename below if 'yolo25n.pt' differs in your
# Ultralytics version (e.g. release naming may vary) — this will auto-download
# the COCO-pretrained checkpoint on first use.
from ultralytics import YOLO

model = YOLO('yolo25n.pt')

print(f"Task: {model.task}")
print(f"Number of classes (nc): {len(model.names)}")
print("Class names (COCO order, index -> name):")
for i, n in model.names.items():
    print(f"  {i:2d}: {n}")


**Mismatch to resolve:** the model natively expects **80 COCO classes** in a
specific index order; VOC provides **20 classes** with different names and a
different index order (see the correspondence table). This is what the next
section aligns before we can run a meaningful baseline `val()`.


## 4. Class Alignment — VOC index → COCO index

In [ ]:
# Name differences between VOC and COCO (from the agreed correspondence table)
VOC_TO_COCO_NAME = {
    'aeroplane':   'airplane',
    'bicycle':     'bicycle',
    'bird':        'bird',
    'boat':        'boat',
    'bottle':      'bottle',
    'bus':         'bus',
    'car':         'car',
    'cat':         'cat',
    'chair':       'chair',
    'cow':         'cow',
    'diningtable': 'dining table',
    'dog':         'dog',
    'horse':       'horse',
    'motorbike':   'motorcycle',
    'person':      'person',
    'pottedplant': 'potted plant',
    'sheep':       'sheep',
    'sofa':        'couch',
    'train':       'train',
    'tvmonitor':   'tv',
}

def build_index_map(voc_names: dict, coco_names: dict) -> dict:
    """Map VOC class index -> COCO class index by shared meaning.
    Fails loudly (never silently drops a class) if any VOC class can't be resolved.
    """
    coco_name_to_idx = {name: idx for idx, name in coco_names.items()}
    index_map = {}
    for voc_idx, voc_name in voc_names.items():
        coco_name = VOC_TO_COCO_NAME.get(voc_name)
        if coco_name is None or coco_name not in coco_name_to_idx:
            raise ValueError(f"No COCO match found for VOC class '{voc_name}' (idx {voc_idx})")
        index_map[voc_idx] = coco_name_to_idx[coco_name]
    return index_map


voc_names = voc_data['names']   # {idx: name}, 20 classes
coco_names = model.names        # {idx: name}, 80 classes

index_map = build_index_map(voc_names, coco_names)
assert len(index_map) == 20, f"Expected 20 mapped classes, got {len(index_map)}"

print("VOC idx -> COCO idx   (VOC name -> COCO name)")
for v_idx, c_idx in index_map.items():
    print(f"  {v_idx:2d} -> {c_idx:2d}   ({voc_names[v_idx]} -> {coco_names[c_idx]})")


### 4a. Build a remapped val split (labels rewritten, images symlinked)

In [ ]:
def remap_val_labels(voc_val_img_dir, index_map, output_root):
    """
    Create a parallel val split under output_root with:
      - images/  : symlinked to the originals (no duplication, no Drive sync)
      - labels/  : same box geometry, class id rewritten VOC idx -> COCO idx
    """
    voc_val_img_dir = Path(voc_val_img_dir if not isinstance(voc_val_img_dir, list) else voc_val_img_dir[0])
    voc_val_label_dir = Path(str(voc_val_img_dir).replace('images', 'labels'))

    out_img_dir = Path(output_root) / 'images'
    out_label_dir = Path(output_root) / 'labels'
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_label_dir.mkdir(parents=True, exist_ok=True)

    n_images, n_labels = 0, 0
    for img_path in voc_val_img_dir.glob('*.jpg'):
        link_path = out_img_dir / img_path.name
        if not link_path.exists():
            link_path.symlink_to(img_path.resolve())
        n_images += 1

        label_path = voc_val_label_dir / f'{img_path.stem}.txt'
        out_label_path = out_label_dir / f'{img_path.stem}.txt'
        if label_path.exists():
            lines = []
            for line in label_path.read_text().splitlines():
                parts = line.split()
                voc_cls = int(parts[0])
                parts[0] = str(index_map[voc_cls])
                lines.append(' '.join(parts))
            out_label_path.write_text('\n'.join(lines))
            n_labels += 1

    return out_img_dir, n_images, n_labels


# Ephemeral, local to the Colab session — cheap to regenerate from the cell above.
REMAP_ROOT = '/content/voc_val_coco_idx'
remapped_img_dir, n_img, n_lbl = remap_val_labels(voc_data['val'], index_map, REMAP_ROOT)
print(f"Remapped val split: {n_img} images, {n_lbl} label files -> {remapped_img_dir}")


### 4b. Write an eval-only data yaml (nc=80, COCO-indexed labels)

In [ ]:
import yaml

eval_yaml_path = f'{PROJECT_ROOT}/configs/voc_val_coco_idx.yaml'
eval_yaml_content = {
    'path': str(Path(REMAP_ROOT)),
    'val': 'images',
    'nc': 80,
    'names': coco_names,
}
with open(eval_yaml_path, 'w') as f:
    yaml.safe_dump(eval_yaml_content, f, sort_keys=False)

print(f"Eval-only yaml written to: {eval_yaml_path}\n")
print(Path(eval_yaml_path).read_text())


## 5. Baseline Evaluation (zero-shot)

Runs the COCO-pretrained `yolo25n` — untouched, native 80-class head — against
VOC's val split with COCO-remapped labels. This is the "before fine-tuning"
number V2 will compare against.


In [ ]:
baseline_results = model.val(
    data=eval_yaml_path,
    split='val',
    imgsz=640,
    project=f'{PROJECT_ROOT}/runs',
    name='baseline_zero_shot',
)


### 5a. Extract metrics — native 80-class AND shared-20-class views

In [ ]:
def extract_metrics(results, index_map, voc_names, coco_names):
    native = {
        'map50': float(results.box.map50),
        'map50_95': float(results.box.map),
    }

    per_class_map5095 = results.box.maps                       # array[nc=80], 0 where no GT
    ap_class_index = list(results.box.ap_class_index)           # classes actually present
    ap50_by_present_class = dict(zip(ap_class_index, results.box.ap50))

    per_class_rows = []
    map5095_values, map50_values = [], []
    for voc_idx, coco_idx in index_map.items():
        m5095 = float(per_class_map5095[coco_idx])
        m50 = float(ap50_by_present_class.get(coco_idx, 0.0))
        per_class_rows.append({
            'voc_class': voc_names[voc_idx],
            'coco_class': coco_names[coco_idx],
            'AP50': m50,
            'AP50-95': m5095,
        })
        map5095_values.append(m5095)
        map50_values.append(m50)

    shared = {
        'mAP50_shared_20': sum(map50_values) / len(map50_values),
        'mAP50-95_shared_20': sum(map5095_values) / len(map5095_values),
    }

    return {'native_80class': native, 'shared_20class': shared, 'per_class': per_class_rows}


baseline_metrics = extract_metrics(baseline_results, index_map, voc_names, coco_names)

print("Native (80-class head, as Ultralytics reports it):")
print(f"  mAP50    = {baseline_metrics['native_80class']['map50']:.4f}")
print(f"  mAP50-95 = {baseline_metrics['native_80class']['map50_95']:.4f}\n")

print("Shared-classes-only (20 VOC-relevant classes):")
print(f"  mAP50    = {baseline_metrics['shared_20class']['mAP50_shared_20']:.4f}")
print(f"  mAP50-95 = {baseline_metrics['shared_20class']['mAP50-95_shared_20']:.4f}\n")

print("Per-class breakdown:")
for row in baseline_metrics['per_class']:
    print(f"  {row['voc_class']:15s} (~{row['coco_class']:15s})  "
          f"AP50={row['AP50']:.3f}  AP50-95={row['AP50-95']:.3f}")


### 5b. Persist baseline report to Drive

In [ ]:
import json
from datetime import datetime

report_path = f"{PROJECT_ROOT}/reports/baseline_{datetime.now():%Y%m%d_%H%M%S}.json"
with open(report_path, 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

print(f"Baseline report saved: {report_path}")


## V1 complete

We now have:
- A verified, working VOC dataset in YOLO format
- Confirmed dataset/model alignment (VOC idx -> COCO idx mapping)
- A **baseline zero-shot report** persisted to Drive (`reports/baseline_*.json`)

**Deferred to V2** (separate notebook, not implemented here):
- Fine-tune `yolo25n` on native VOC (`nc=20`, backbone transfer-learned from COCO weights)
- Validate the fine-tuned model natively (no remapping needed)
- Load this baseline report and diff it against the fine-tuned results
